In [1]:
"""
Week 5 — Model Training, Evaluation & Comparison
Phishing URL Detection Dataset
Models: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting,
        AdaBoost, SVM, KNN, Naive Bayes, MLP Neural Network
Extras: 5-Fold CV, Hyperparameter Tuning (Best Model), Feature Importance
"""

import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier,
                               AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, matthews_corrcoef,
                              confusion_matrix, classification_report)
from sklearn.model_selection import (StratifiedKFold, cross_val_score,
                                     RandomizedSearchCV)

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("STEP 1: LOADING REFINED DATASET")
print("=" * 70)

df_train = pd.read_csv("./Week4/train_refined.csv")
df_test  = pd.read_csv("./Week4/test_refined.csv")

X_train = df_train.drop(columns=["status"])
y_train = df_train["status"]
X_test  = df_test.drop(columns=["status"])
y_test  = df_test["status"]

print(f"  Training set : {X_train.shape[0]} samples x {X_train.shape[1]} features")
print(f"  Test set     : {X_test.shape[0]} samples x {X_test.shape[1]} features")
print(f"  Train class  : {y_train.value_counts().to_dict()}")
print(f"  Test class   : {y_test.value_counts().to_dict()}")

STEP 1: LOADING REFINED DATASET
  Training set : 9144 samples x 99 features
  Test set     : 2286 samples x 99 features
  Train class  : {0: 4572, 1: 4572}
  Test class   : {1: 1143, 0: 1143}


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. DEFINE MODELS
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 2: MODEL DEFINITIONS")
print("=" * 70)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=200, random_state=42),
    "AdaBoost":            AdaBoostClassifier(n_estimators=200, random_state=42),
    "SVM (RBF)":           SVC(kernel="rbf", probability=True, random_state=42),
    "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    "Naive Bayes":         GaussianNB(),
    "MLP Neural Network":  MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500,
                                         random_state=42),
}

print(f"  {len(models)} models registered:")
for name in models:
    print(f"    - {name}")


STEP 2: MODEL DEFINITIONS
  9 models registered:
    - Logistic Regression
    - Decision Tree
    - Random Forest
    - Gradient Boosting
    - AdaBoost
    - SVM (RBF)
    - KNN (k=5)
    - Naive Bayes
    - MLP Neural Network


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. TRAIN & EVALUATE ALL MODELS
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 3: TRAINING AND EVALUATION ON TEST SET")
print("=" * 70)

results = {}
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"\n  {'Model':<22} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} "
      f"{'AUC':>7} {'MCC':>7} {'Time':>8}")
print("  " + "-" * 72)

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    y_pred  = model.predict(X_test)
    y_proba = (model.predict_proba(X_test)[:, 1]
               if hasattr(model, "predict_proba")
               else model.decision_function(X_test))

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    auc  = roc_auc_score(y_test, y_proba)
    mcc  = matthews_corrcoef(y_test, y_pred)

    cv_f1 = cross_val_score(model, X_train, y_train,
                            cv=cv_splitter, scoring="f1", n_jobs=-1)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    results[name] = {
        "accuracy": acc, "precision": prec, "recall": rec,
        "f1": f1, "auc": auc, "mcc": mcc,
        "cv_f1_mean": cv_f1.mean(), "cv_f1_std": cv_f1.std(),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
        "train_time": train_time,
        "y_pred": y_pred, "y_proba": y_proba,
    }
    print(f"  {name:<22} {acc:>7.4f} {prec:>7.4f} {rec:>7.4f} {f1:>7.4f} "
          f"{auc:>7.4f} {mcc:>7.4f} {train_time:>6.2f}s")


STEP 3: TRAINING AND EVALUATION ON TEST SET

  Model                      Acc    Prec     Rec      F1     AUC     MCC     Time
  ------------------------------------------------------------------------
  Logistic Regression     0.9405  0.9382  0.9431  0.9407  0.9842  0.8810   0.33s
  Decision Tree           0.9300  0.9293  0.9309  0.9301  0.9300  0.8600   0.28s
  Random Forest           0.9619  0.9583  0.9659  0.9621  0.9937  0.9239   0.74s
  Gradient Boosting       0.9567  0.9531  0.9606  0.9569  0.9922  0.9134  12.60s
  AdaBoost                0.9366  0.9393  0.9335  0.9364  0.9858  0.8732   5.62s
  SVM (RBF)               0.9567  0.9563  0.9571  0.9567  0.9896  0.9134   9.66s
  KNN (k=5)               0.9458  0.9529  0.9379  0.9453  0.9780  0.8916   0.01s
  Naive Bayes             0.6868  0.9162  0.4112  0.5676  0.9186  0.4477   0.02s
  MLP Neural Network      0.9576  0.9580  0.9571  0.9575  0.9900  0.9151  13.49s


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. 5-FOLD CROSS-VALIDATION SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 4: 5-FOLD CROSS-VALIDATION F1 SCORES (TRAINING SET)")
print("=" * 70)
print(f"\n  {'Model':<22} {'CV F1 Mean':>12} {'CV F1 Std':>12}")
print("  " + "-" * 48)
for name, r in results.items():
    print(f"  {name:<22} {r['cv_f1_mean']:>12.4f} {r['cv_f1_std']:>12.4f}")


STEP 4: 5-FOLD CROSS-VALIDATION F1 SCORES (TRAINING SET)

  Model                    CV F1 Mean    CV F1 Std
  ------------------------------------------------
  Logistic Regression          0.9519       0.0067
  Decision Tree                0.9345       0.0075
  Random Forest                0.9657       0.0072
  Gradient Boosting            0.9640       0.0088
  AdaBoost                     0.9499       0.0082
  SVM (RBF)                    0.9593       0.0063
  KNN (k=5)                    0.9383       0.0065
  Naive Bayes                  0.5578       0.0302
  MLP Neural Network           0.9595       0.0060


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. CONFUSION MATRICES
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 5: CONFUSION MATRICES")
print("=" * 70)
print(f"\n  {'Model':<22} {'TN':>6} {'FP':>6} {'FN':>6} {'TP':>6}  "
      f"{'FPR':>8} {'FNR':>8}")
print("  " + "-" * 68)
for name, r in results.items():
    tn, fp, fn, tp = r['tn'], r['fp'], r['fn'], r['tp']
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    print(f"  {name:<22} {tn:>6} {fp:>6} {fn:>6} {tp:>6}  "
          f"{fpr:>8.4f} {fnr:>8.4f}")


STEP 5: CONFUSION MATRICES

  Model                      TN     FP     FN     TP       FPR      FNR
  --------------------------------------------------------------------
  Logistic Regression      1072     71     65   1078    0.0621   0.0569
  Decision Tree            1062     81     79   1064    0.0709   0.0691
  Random Forest            1095     48     39   1104    0.0420   0.0341
  Gradient Boosting        1089     54     45   1098    0.0472   0.0394
  AdaBoost                 1074     69     76   1067    0.0604   0.0665
  SVM (RBF)                1093     50     49   1094    0.0437   0.0429
  KNN (k=5)                1090     53     71   1072    0.0464   0.0621
  Naive Bayes              1100     43    673    470    0.0376   0.5888
  MLP Neural Network       1095     48     49   1094    0.0420   0.0429


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# 6. DETAILED CLASSIFICATION REPORTS
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 6: DETAILED CLASSIFICATION REPORTS")
print("=" * 70)
for name, r in results.items():
    print(f"\n  -- {name} --")
    print(classification_report(y_test, r["y_pred"],
                                target_names=["legitimate", "phishing"],
                                digits=4))


STEP 6: DETAILED CLASSIFICATION REPORTS

  -- Logistic Regression --
              precision    recall  f1-score   support

  legitimate     0.9428    0.9379    0.9404      1143
    phishing     0.9382    0.9431    0.9407      1143

    accuracy                         0.9405      2286
   macro avg     0.9405    0.9405    0.9405      2286
weighted avg     0.9405    0.9405    0.9405      2286


  -- Decision Tree --
              precision    recall  f1-score   support

  legitimate     0.9308    0.9291    0.9299      1143
    phishing     0.9293    0.9309    0.9301      1143

    accuracy                         0.9300      2286
   macro avg     0.9300    0.9300    0.9300      2286
weighted avg     0.9300    0.9300    0.9300      2286


  -- Random Forest --
              precision    recall  f1-score   support

  legitimate     0.9656    0.9580    0.9618      1143
    phishing     0.9583    0.9659    0.9621      1143

    accuracy                         0.9619      2286
   macro avg

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# 7. RANKED COMPARISON TABLE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 7: RANKED MODEL COMPARISON (by F1-Score)")
print("=" * 70)
ranked = sorted(results.items(), key=lambda x: x[1]["f1"], reverse=True)
print(f"\n  {'Rank':<5} {'Model':<22} {'Accuracy':>10} {'Precision':>10} "
      f"{'Recall':>8} {'F1':>8} {'AUC-ROC':>9} {'MCC':>8}")
print("  " + "-" * 84)
for i, (name, r) in enumerate(ranked, 1):
    print(f"  {i:<5} {name:<22} {r['accuracy']:>10.4f} {r['precision']:>10.4f} "
          f"{r['recall']:>8.4f} {r['f1']:>8.4f} {r['auc']:>9.4f} {r['mcc']:>8.4f}")

best_name = ranked[0][0]
print(f"\n  Best Model: {best_name}")
print(f"  F1 = {results[best_name]['f1']:.4f}  AUC = {results[best_name]['auc']:.4f}  "
      f"MCC = {results[best_name]['mcc']:.4f}")


STEP 7: RANKED MODEL COMPARISON (by F1-Score)

  Rank  Model                    Accuracy  Precision   Recall       F1   AUC-ROC      MCC
  ------------------------------------------------------------------------------------
  1     Random Forest              0.9619     0.9583   0.9659   0.9621    0.9937   0.9239
  2     MLP Neural Network         0.9576     0.9580   0.9571   0.9575    0.9900   0.9151
  3     Gradient Boosting          0.9567     0.9531   0.9606   0.9569    0.9922   0.9134
  4     SVM (RBF)                  0.9567     0.9563   0.9571   0.9567    0.9896   0.9134
  5     KNN (k=5)                  0.9458     0.9529   0.9379   0.9453    0.9780   0.8916
  6     Logistic Regression        0.9405     0.9382   0.9431   0.9407    0.9842   0.8810
  7     AdaBoost                   0.9366     0.9393   0.9335   0.9364    0.9858   0.8732
  8     Decision Tree              0.9300     0.9293   0.9309   0.9301    0.9300   0.8600
  9     Naive Bayes                0.6868     0.9162   

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# 8. HYPERPARAMETER TUNING — RANDOM FOREST
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 8: HYPERPARAMETER TUNING — RANDOM FOREST (RandomizedSearchCV)")
print("=" * 70)

param_dist = {
    "n_estimators":      [200, 300, 400],
    "max_depth":         [None, 20, 30],
    "min_samples_split": [2, 5],
    "max_features":      ["sqrt", 0.3],
}
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist, n_iter=10, cv=cv3, scoring="f1",
    n_jobs=-1, random_state=42, verbose=0
)
search.fit(X_train, y_train)
print(f"\n  Best hyperparameters : {search.best_params_}")
print(f"  Best CV F1 (3-fold)  : {search.best_score_:.4f}")

best_rf    = search.best_estimator_
y_pred_t   = best_rf.predict(X_test)
y_proba_t  = best_rf.predict_proba(X_test)[:, 1]

acc_t  = accuracy_score(y_test, y_pred_t)
prec_t = precision_score(y_test, y_pred_t)
rec_t  = recall_score(y_test, y_pred_t)
f1_t   = f1_score(y_test, y_pred_t)
auc_t  = roc_auc_score(y_test, y_proba_t)
mcc_t  = matthews_corrcoef(y_test, y_pred_t)
tn_t, fp_t, fn_t, tp_t = confusion_matrix(y_test, y_pred_t).ravel()

print(f"\n  {'Metric':<12} {'Default RF':>12} {'Tuned RF':>12} {'Delta':>10}")
print("  " + "-" * 48)
for metric, dval, tval in [
    ("Accuracy",  results["Random Forest"]["accuracy"],  acc_t),
    ("Precision", results["Random Forest"]["precision"], prec_t),
    ("Recall",    results["Random Forest"]["recall"],    rec_t),
    ("F1-Score",  results["Random Forest"]["f1"],        f1_t),
    ("AUC-ROC",   results["Random Forest"]["auc"],       auc_t),
    ("MCC",       results["Random Forest"]["mcc"],       mcc_t),
]:
    print(f"  {metric:<12} {dval:>12.4f} {tval:>12.4f} {tval-dval:>+10.4f}")

print(f"\n  Confusion Matrix  : TN={tn_t}  FP={fp_t}  FN={fn_t}  TP={tp_t}")
print(f"\n  Classification Report:")
print(classification_report(y_test, y_pred_t,
                             target_names=["legitimate", "phishing"], digits=4))


STEP 8: HYPERPARAMETER TUNING — RANDOM FOREST (RandomizedSearchCV)

  Best hyperparameters : {'n_estimators': 300, 'min_samples_split': 2, 'max_features': 'sqrt', 'max_depth': 20}
  Best CV F1 (3-fold)  : 0.9659

  Metric         Default RF     Tuned RF      Delta
  ------------------------------------------------
  Accuracy           0.9619       0.9611    -0.0009
  Precision          0.9583       0.9567    -0.0017
  Recall             0.9659       0.9659    +0.0000
  F1-Score           0.9621       0.9613    -0.0008
  AUC-ROC            0.9937       0.9937    +0.0000
  MCC                0.9239       0.9222    -0.0017

  Confusion Matrix  : TN=1093  FP=50  FN=39  TP=1104

  Classification Report:
              precision    recall  f1-score   support

  legitimate     0.9655    0.9563    0.9609      1143
    phishing     0.9567    0.9659    0.9613      1143

    accuracy                         0.9611      2286
   macro avg     0.9611    0.9611    0.9611      2286
weighted avg     0.

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# 9. FEATURE IMPORTANCE — TUNED RANDOM FOREST
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("STEP 9: FEATURE IMPORTANCE — TUNED RANDOM FOREST (Top 25)")
print("=" * 70)
fi = pd.Series(best_rf.feature_importances_,
               index=X_train.columns).sort_values(ascending=False)
print(f"\n  {'Rank':<5} {'Feature':<35} {'Importance':>12} {'Cumulative':>12}")
print("  " + "-" * 68)
cumsum = 0
for rank, (feat, imp) in enumerate(fi.head(25).items(), 1):
    cumsum += imp
    print(f"  {rank:<5} {feat:<35} {imp:>12.5f} {cumsum:>11.1%}")

print(f"\n  Top 10 features explain : {fi.head(10).sum():.1%} of total importance")
print(f"  Top 25 features explain : {fi.head(25).sum():.1%} of total importance")

STEP 9: FEATURE IMPORTANCE — TUNED RANDOM FOREST (Top 25)

  Rank  Feature                               Importance   Cumulative
  --------------------------------------------------------------------
  1     google_index                             0.16367       16.4%
  2     page_rank                                0.09600       26.0%
  3     nb_hyperlinks                            0.06969       32.9%
  4     web_traffic                              0.06842       39.8%
  5     nb_www                                   0.03831       43.6%
  6     ratio_extHyperlinks                      0.02832       46.4%
  7     domain_age                               0.02501       48.9%
  8     longest_word_path                        0.02499       51.4%
  9     ratio_intHyperlinks                      0.02455       53.9%
  10    safe_anchor                              0.02192       56.1%
  11    hyperlink_diversity                      0.01959       58.0%
  12    phish_hints                      

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 10. SAVE RESULTS
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("STEP 10: SAVING RESULTS")
print("=" * 70)

rows = []
for name, r in results.items():
    rows.append({
        "Model": name, "Accuracy": round(r["accuracy"], 4),
        "Precision": round(r["precision"], 4), "Recall": round(r["recall"], 4),
        "F1_Score": round(r["f1"], 4), "AUC_ROC": round(r["auc"], 4),
        "MCC": round(r["mcc"], 4), "CV_F1_Mean": round(r["cv_f1_mean"], 4),
        "CV_F1_Std": round(r["cv_f1_std"], 4),
        "TN": r["tn"], "FP": r["fp"], "FN": r["fn"], "TP": r["tp"],
        "Train_Time_s": round(r["train_time"], 2),
    })
rows.append({
    "Model": "Random Forest (Tuned)", "Accuracy": round(acc_t, 4),
    "Precision": round(prec_t, 4), "Recall": round(rec_t, 4),
    "F1_Score": round(f1_t, 4), "AUC_ROC": round(auc_t, 4),
    "MCC": round(mcc_t, 4), "CV_F1_Mean": round(search.best_score_, 4),
    "CV_F1_Std": 0, "TN": tn_t, "FP": fp_t, "FN": fn_t, "TP": tp_t,
    "Train_Time_s": None,
})
cmp_df = pd.DataFrame(rows).sort_values("F1_Score", ascending=False)
cmp_df.to_csv("./Week5-6/model_comparison.csv", index=False)
print("  Saved: model_comparison.csv")

fi_df = fi.reset_index()
fi_df.columns = ["Feature", "Importance"]
fi_df["Cumulative"] = fi_df["Importance"].cumsum()
fi_df.to_csv("./Week5-6/feature_importance_rf_tuned.csv", index=False)
print("  Saved: feature_importance_rf_tuned.csv")

print("\n" + "=" * 70)
print("WEEK 5 COMPLETE")
print("=" * 70)
print(f"  Models trained      : {len(models)}")
print(f"  Best baseline F1    : Random Forest ({results['Random Forest']['f1']:.4f})")
print(f"  Best tuned F1       : Random Forest Tuned ({f1_t:.4f})")
print(f"  Best AUC-ROC        : {max(r['auc'] for r in results.values()):.4f}")
print(f"  Outputs: model_comparison.csv, feature_importance_rf_tuned.csv")


STEP 10: SAVING RESULTS
  Saved: model_comparison.csv
  Saved: feature_importance_rf_tuned.csv

WEEK 5 COMPLETE
  Models trained      : 9
  Best baseline F1    : Random Forest (0.9621)
  Best tuned F1       : Random Forest Tuned (0.9613)
  Best AUC-ROC        : 0.9937
  Outputs: model_comparison.csv, feature_importance_rf_tuned.csv
